# Notebook 13 — EDA Report
### Sprint 4 | Data Inspection & Exploratory Data Analysis (EDA)

This notebook consolidates every finding from Notebooks 1-12 into a single, complete EDA
summary. **No preprocessing is performed here** — this notebook documents problems and
recommendations only, per the sprint brief. The next sprint (Data Cleaning &
Preprocessing) is where these recommendations get implemented.

Every number below is re-verified directly against the dataset in this notebook, not
copied from memory of earlier notebooks.


In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("telco_churn.csv")
df_clean = df.copy()
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0)

print(f"Report generated against: {df.shape[0]:,} rows, {df.shape[1]} columns")


Report generated against: 7,043 rows, 21 columns


---
## 1. Dataset Overview

| Field | Detail |
|---|---|
| **Dataset name** | Telco Customer Churn |
| **Source** | IBM Sample Data Sets, via GitHub mirror (`IBM/telco-customer-churn-on-icp4d`) |
| **Domain** | Telecommunications — Customer Churn |
| **Size** | 7,043 rows (customers) x 21 columns |
| **Features** | 19 candidate input features: demographics (gender, SeniorCitizen, Partner, Dependents), account info (tenure, Contract, PaperlessBilling, PaymentMethod), services (PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies), billing (MonthlyCharges, TotalCharges) |
| **Target** | `Churn` (Yes/No) — binary classification |
| **Identifier** | `customerID` (excluded from modeling) |


In [2]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Target: Churn -> {dict(df['Churn'].value_counts())}")


Rows: 7,043
Columns: 21
Target: Churn -> {'No': np.int64(5174), 'Yes': np.int64(1869)}


---
## 2. Data Quality

### 2.1 Missing Values


In [3]:
blank_total_charges = (df['TotalCharges'].str.strip() == '').sum()
print(f"Values reported as missing by df.isnull().sum(): {df.isnull().sum().sum()}")
print(f"ACTUAL missing values (blank strings in TotalCharges, invisible to .isnull()): {blank_total_charges}")
print(f"Percentage of dataset affected: {blank_total_charges/len(df)*100:.2f}%")
print(f"\nAll {blank_total_charges} affected customers have tenure == 0:",
      (df.loc[df['TotalCharges'].str.strip()=='', 'tenure'] == 0).all())


Values reported as missing by df.isnull().sum(): 0
ACTUAL missing values (blank strings in TotalCharges, invisible to .isnull()): 11
Percentage of dataset affected: 0.16%

All 11 affected customers have tenure == 0: True


**Finding:** `df.isnull().sum()` reports zero missing values across the entire
dataset — misleadingly. `TotalCharges` is stored as text and contains 11 blank-string
entries (0.16% of the dataset), invisible to standard missing-value detection. All 11
affected customers have `tenure == 0` — they are brand-new customers who genuinely have
not been billed yet, not a data-entry error.

### 2.2 Duplicates


In [4]:
full_duplicates = df.duplicated().sum()
duplicate_ids = df['customerID'].duplicated().sum()
print(f"Fully duplicated rows: {full_duplicates}")
print(f"Duplicate customerID values: {duplicate_ids}")


Fully duplicated rows: 0
Duplicate customerID values: 0


**Finding:** Zero fully duplicated rows and zero duplicate `customerID` values —
this dataset has no duplicate-record problem.

### 2.3 Invalid Values


In [5]:
negative_checks = {
    'tenure': (df['tenure'] < 0).sum(),
    'MonthlyCharges': (df['MonthlyCharges'] < 0).sum(),
    'TotalCharges': (df_clean['TotalCharges'] < 0).sum(),
}
print("Negative values found:", negative_checks)

expected_categories = {
    'gender': {'Male', 'Female'},
    'Churn': {'Yes', 'No'},
    'Contract': {'Month-to-month', 'One year', 'Two year'},
}
for col, expected in expected_categories.items():
    unexpected = set(df[col].unique()) - expected
    print(f"{col}: unexpected categories = {unexpected if unexpected else 'None'}")


Negative values found: {'tenure': np.int64(0), 'MonthlyCharges': np.int64(0), 'TotalCharges': np.int64(0)}
gender: unexpected categories = None
Churn: unexpected categories = None
Contract: unexpected categories = None


**Finding:** No negative values in any numeric column; no unexpected category
labels in the spot-checked categorical columns. The only genuine invalid-value issue in
this dataset is the disguised missing data covered in 2.1.

### 2.4 Data-Type Issues


In [6]:
print(df.dtypes.value_counts())
print("\nColumn requiring correction: TotalCharges (object -> should be float64)")


str        18
int64       2
float64     1
Name: count, dtype: int64

Column requiring correction: TotalCharges (object -> should be float64)


**Finding:** `TotalCharges` is the only column with a data-type mismatch — stored as
`object` (text) despite representing a currency amount. This is the direct cause of both
the disguised missing values (2.1) and its exclusion from default numeric summaries
(Notebook 2).


---
## 3. Statistical Findings

### 3.1 Central Tendency & Dispersion


In [7]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
summary = df_clean[numeric_cols].agg(['mean', 'median', 'std', 'var']).round(2)
print(summary)


        tenure  MonthlyCharges  TotalCharges
mean     32.37           64.76       2279.73
median   29.00           70.35       1394.55
std      24.56           30.09       2266.79
var     603.17          905.41    5138357.17


**Finding:** `tenure` and `MonthlyCharges` show reasonably close mean/median pairs
(no strong skew); `TotalCharges`'s mean (\$2,279.73) sits well above its median
(\$1,394.55), an early signal of its right skew, confirmed below.

### 3.2 Distribution & Skewness


In [8]:
for col in numeric_cols:
    skew = stats.skew(df_clean[col])
    kurt = stats.kurtosis(df_clean[col])
    print(f"{col}: skew={skew:.2f}, kurtosis={kurt:.2f}")


tenure: skew=0.24, kurtosis=-1.39
MonthlyCharges: skew=-0.22, kurtosis=-1.26
TotalCharges: skew=0.96, kurtosis=-0.23


**Finding:** None of the three numeric features are normally distributed
(Shapiro-Wilk confirmed this formally in Notebook 8 for all three). `TotalCharges` is the
only meaningfully skewed feature (skew ≈ 0.96, right-skewed); `tenure` and
`MonthlyCharges` are roughly symmetric but notably flat/multi-clustered rather than
bell-shaped — `tenure` specifically has a sharp spike at very low tenure values.

### 3.3 Outliers


In [9]:
for col in numeric_cols:
    Q1, Q3 = df_clean[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    outliers = df_clean[(df_clean[col] < Q1 - 1.5*IQR) | (df_clean[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers)} univariate IQR outliers")

df_clean['residual'] = df_clean['TotalCharges'] - (df_clean['tenure'] * df_clean['MonthlyCharges'])
multivariate_outliers = (np.abs(stats.zscore(df_clean['residual'])) > 3).sum()
print(f"\nMultivariate outliers (TotalCharges vs tenure x MonthlyCharges): {multivariate_outliers}")


tenure: 0 univariate IQR outliers
MonthlyCharges: 0 univariate IQR outliers
TotalCharges: 0 univariate IQR outliers

Multivariate outliers (TotalCharges vs tenure x MonthlyCharges): 112


**Finding:** Zero univariate outliers in any numeric column, by both IQR and
Z-score methods (Notebook 7). However, 112 genuine multivariate outliers exist —
customers whose `TotalCharges` doesn't match a simple constant-rate assumption, most
plausibly reflecting real historical billing/plan changes. **Decision: retain these
records; the residual is a candidate engineered feature, not a value to remove.**


---
## 4. Relationship Findings

### 4.1 Correlations


In [10]:
df_clean['Churn_numeric'] = (df_clean['Churn'] == 'Yes').astype(int)
corr = df_clean[numeric_cols + ['Churn_numeric']].corr()['Churn_numeric'].drop('Churn_numeric')
print(corr.round(3))


tenure           -0.352
MonthlyCharges    0.193
TotalCharges     -0.198
Name: Churn_numeric, dtype: float64


**Finding:** `tenure` has the strongest linear relationship with churn among numeric
features (-0.35), followed by `MonthlyCharges` (+0.19) and `TotalCharges` (-0.20).
`tenure` and `TotalCharges` are themselves strongly correlated (+0.83, VIF ≈ 8.1) — a
flagged multicollinearity concern for the next sprint.

### 4.2 Feature Relationships (Categorical)


In [11]:
top_categorical_predictors = {}
for col in ['Contract', 'InternetService', 'OnlineSecurity', 'TechSupport', 'PaymentMethod']:
    rates = df_clean.groupby(col)['Churn'].apply(lambda s: (s=='Yes').mean()*100)
    top_categorical_predictors[col] = round(rates.max() - rates.min(), 1)
print(pd.Series(top_categorical_predictors).sort_values(ascending=False))


Contract           39.9
InternetService    34.5
OnlineSecurity     34.4
TechSupport        34.2
PaymentMethod      30.0
dtype: float64


**Finding:** `Contract` (39.9-point spread), `InternetService` (34.5), and
`OnlineSecurity`/`TechSupport` (~34 each) are this dataset's strongest categorical churn
predictors — all substantially stronger than any single numeric feature's correlation.

### 4.3 Important Patterns


- **Month-to-month + Fiber optic** is the single highest-risk customer segment
  (54.6% churn, 30.2% of the customer base — Notebook 12).
- Churn risk is heavily front-loaded: ~56% in the first 3 months, falling to single
  digits past 5 years of tenure (Notebook 11).
- Low-cost, no-internet customers are this dataset's most loyal segment (7.4% churn,
  21.7% of the base).


---
## 5. Visualization Findings

Summarized from Notebook 11's ten visualizations:

1. **Log-transforming `TotalCharges`** overcorrects into mild left skew due to 11
   zero-charge new customers — a naive transform recommendation would have missed this.
2. **`MonthlyCharges` differs only modestly across `Contract` types** — confirming
   `Contract`'s churn relationship is about commitment, not primarily price.
3. **Fiber optic generates the highest average `TotalCharges`** ($3,205) — the
   highest-churn segment is simultaneously the highest-value one.
4. **The retention curve (churn rate by tenure month)** is this sprint's most visually
   striking finding: a steep early cliff, not a gradual decline.
5. **The Contract x InternetService heatmap** pinpoints the exact highest-risk
   combination (54.6% churn) that neither factor alone fully reveals.


---
## 6. Potential Data Problems for the Next Sprint

| # | Problem | Severity | Affected |
|---|---|---|---|
| 1 | `TotalCharges` stored as text with 11 disguised blank values | Low volume, but breaks numeric operations if unaddressed | 11 rows (0.16%) |
| 2 | `TotalCharges` multicollinear with `tenure` x `MonthlyCharges` (VIF ≈ 8.1) | Moderate — affects linear models specifically | All rows |
| 3 | 112 multivariate billing-residual outliers | Low — genuine data, not an error, but worth flagging as a feature | 112 rows (1.6%) |
| 4 | `Churn` target is moderately imbalanced (73.5%/26.5%) | Moderate — affects metric choice and split strategy | All rows |
| 5 | `TotalCharges` is right-skewed (skew ≈ 0.96) | Low-moderate — affects linear model assumptions | All rows |
| 6 | 16 categorical columns require encoding before modeling | Expected, not a defect | All rows |


---
## 7. Recommendations for the Next Sprint (Data Cleaning & Preprocessing)

### Missing-Value Treatment
- Convert `TotalCharges` to numeric (`pd.to_numeric(..., errors='coerce')`).
- Fill the 11 resulting `NaN` values with **0**, not mean/median — justified because all
  11 belong to `tenure == 0` customers who genuinely haven't been billed yet.

### Outlier Treatment
- **Do not remove** any of the 112 multivariate residual outliers — retain as genuine
  observations.
- Consider engineering `residual = TotalCharges - (tenure * MonthlyCharges)` as a new
  feature capturing historical billing changes.

### Encoding
- One-hot encode all 16 categorical columns (all confirmed low-cardinality, Notebook 9;
  no rare categories requiring consolidation).
- Binary-encode `Churn` (Yes=1, No=0) for the target.

### Scaling
- Standardize `tenure` and `MonthlyCharges` (roughly symmetric, safe for mean-based
  scaling).
- Consider `RobustScaler` or a log transform for `TotalCharges`, given its skew — but be
  aware a plain `log1p` transform overcorrects into left skew due to zero-inflation from
  new customers (Notebook 11); evaluate this carefully rather than applying it blindly.

### Feature Transformation
- Evaluate dropping `TotalCharges` in favor of `tenure` and `MonthlyCharges` given the
  VIF ≈ 8.1 multicollinearity finding, OR retain it but monitor coefficient stability if
  using a linear model.

### Feature Engineering
- Engineer a `tenure_group` categorical feature (e.g., 0-6mo, 7-12mo, 13-24mo, 25-48mo,
  49-72mo) — Notebook 12 showed churn rate varies meaningfully and monotonically across
  these bands.
- Engineer the billing `residual` feature described above.
- Consider a combined "protective add-on count" feature summing `OnlineSecurity`,
  `OnlineBackup`, `DeviceProtection`, and `TechSupport` subscriptions, given their shared
  strong relationship with churn (Notebook 9).

### Data-Type Correction
- `TotalCharges`: `object` → `float64` (covered above).
- No other columns require type correction.

### Modeling Strategy Notes (from Notebook 10)
- Use a **stratified** train/test split to preserve the 73.5%/26.5% class ratio.
- Evaluate with precision, recall, F1, and/or ROC-AUC — **not** accuracy alone, given the
  moderate class imbalance.


---
## Report Summary

This EDA sprint examined a 7,043-row, 21-column real-world telecom churn dataset across
13 notebooks, covering dataset understanding, inspection, data quality, univariate,
bivariate, multivariate, outlier, distribution, categorical, and target variable
analysis, visualization, and business insight synthesis. One data-quality issue
(disguised missing values in `TotalCharges`), one multicollinearity concern (`TotalCharges`
vs `tenure`/`MonthlyCharges`), and one genuine-but-flaggable anomaly (112 billing
residual outliers) were identified and documented — **not fixed**, per this sprint's
scope. `Contract`, `InternetService`, `OnlineSecurity`, and `TechSupport` emerged as the
strongest churn predictors, with a specific, quantified highest-risk segment identified
(Month-to-month + Fiber optic, 54.6% churn, 30.2% of the customer base). All
recommendations above are scoped for direct implementation in the next sprint.

**Next notebook:** `14_EDA_Mini_Challenge.ipynb` — an independent, unguided EDA
performed on a new, unfamiliar dataset, demonstrating this same process without
step-by-step instructions.
